In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: O funcție Qiskit de Q-CTRL Fire Opal
*Consultă [referința API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor IBM Quantum&reg; cu plan Premium, plan Flex și plan On-Prem (prin IBM Quantum Platform API). Acestea se află în stadiu de previzualizare și pot suferi modificări.


<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Prezentare generală
Fire Opal Performance Management simplifică obținerea de rezultate relevante de la calculatoarele cuantice la scară pentru oricine, fără a fi nevoie să fii expert în hardware cuantic. Când rulezi circuite cu Fire Opal Performance Management, tehnicile de suprimare a erorilor bazate pe inteligență artificială sunt aplicate automat, permițând scalarea unor probleme mai mari cu mai multe Gate-uri și Qubiți. Această abordare reduce numărul de shot-uri necesare pentru a obține răspunsul corect, fără costuri suplimentare — rezultând economii semnificative atât în timp de calcul, cât și în costuri.

Performance Management suprimă erorile și crește probabilitatea de a obține răspunsul corect pe hardware cu zgomot. Cu alte cuvinte, crește raportul semnal-zgomot. Imaginea următoare arată cum acuratețea sporită permisă de Performance Management poate reduce nevoia de shot-uri suplimentare în cazul unui algoritm Quantum Fourier Transform cu 10 Qubiți. Cu doar 30 de shot-uri, Q-CTRL atinge pragul de încredere de 99%, în timp ce metoda implicită (`QiskitRuntime` Sampler, `optimization_level`=3 și `resilience_level`=1, `ibm_sherbrooke`) necesită 170.000 de shot-uri. Obținând răspunsul corect mai repede, economisești timp de calcul semnificativ.

![Vizualizarea timpului de rulare îmbunătățit](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

Funcția Performance Management poate fi folosită cu orice algoritm și o poți utiliza cu ușurință în locul primitivelor standard [Qiskit Runtime](/guides/primitives). În culise, mai multe tehnici de suprimare a erorilor lucrează împreună pentru a preveni erorile în timpul execuției. Toate metodele din pipeline-ul Fire Opal sunt preconfigurate și independente de algoritm, ceea ce înseamnă că obții întotdeauna cea mai bună performanță din prima.


Pentru a obține acces la Performance Management, [contactează Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descriere
Fire Opal Performance Management are două opțiuni de execuție similare cu primitivele Qiskit Runtime, astfel încât poți înlocui cu ușurință Sampler-ul și Estimator-ul cu variantele Q-CTRL. Fluxul general de utilizare a funcției Performance Management este:
1. Definește-ți Circuit-ul (și operatorii, în cazul Estimator-ului).
2. Rulează Circuit-ul.
3. Recuperează rezultatele.

Pentru a reduce zgomotul hardware, Fire Opal folosește o serie de tehnici de suprimare a erorilor bazate pe inteligență artificială, ilustrate în imaginea următoare. Cu Fire Opal, întregul pipeline este complet automatizat, fără nicio nevoie de configurare.

Pipeline-ul Fire Opal elimină nevoia de costuri suplimentare, cum ar fi creșterea timpului de rulare cuantic sau Qubiți fizici în plus. Rețineți că timpul de procesare clasică rămâne un factor (consultați secțiunea [Benchmark-uri](#benchmarks) pentru estimări, unde „Timp total" reflectă atât procesarea clasică, cât și cea cuantică). Spre deosebire de atenuarea erorilor, care necesită costuri suplimentare sub formă de eșantionare, suprimarea erorilor din Fire Opal funcționează atât la nivel de poartă, cât și la nivel de puls pentru a aborda diverse surse de zgomot și a preveni probabilitatea producerii unei erori. Prin prevenirea erorilor, este eliminată nevoia de post-procesare costisitoare.

Imaginea următoare ilustrează metodele de suprimare a erorilor automatizate de Fire Opal Performance Management.

![Vizualizarea pipeline-ului de suprimare a erorilor](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

Funcția oferă două primitive, Sampler și Estimator, iar intrările și ieșirile ambelor extind specificația implementată pentru [primitivele Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Benchmark-uri
Rezultatele [benchmark-urilor algoritmice publicate](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) demonstrează o îmbunătățire semnificativă a performanței pentru diverși algoritmi, inclusiv Bernstein-Vazirani, transformata Fourier cuantică, căutarea Grover, algoritmul de optimizare aproximativă cuantică și eigensolver-ul variațional cuantic. Restul acestei secțiuni oferă mai multe detalii despre tipurile de algoritmi pe care îi poți rula, precum și performanța și timpii de rulare așteptați.

Următoarele studii independente demonstrează modul în care Performance Management de la Q-CTRL permite cercetarea algoritmică la scară record:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - învățare de kernel cuantic cu până la 50 de Qubiți
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - estimare de fază cuantică cu până la 33 de Qubiți
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - încărcare de date cuantice cu până la 21 de Qubiți

Următorul tabel oferă un ghid aproximativ privind acuratețea și timpii de rulare din benchmark-urile anterioare efectuate pe `ibm_fez`. Performanța pe alte dispozitive poate varia. Timpul de utilizare se bazează pe o presupunere de 10.000 de shot-uri per Circuit. „Numărul de Qubiți" indicat nu reprezintă o limitare strictă, ci praguri aproximative la care poți aștepta o acuratețe a soluției extrem de consistentă. Probleme de dimensiuni mai mari au fost rezolvate cu succes, iar testarea dincolo de aceste limite este încurajată.

| Exemplu    | Număr de Qubiți | Acuratețe | Măsura acurateței | Timp total (s) | Utilizare runtime (s) | Primitivă (Mod) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Rată de succes (Procentul de rulări în care răspunsul corect este șirul de biți cu cel mai mare număr)     | 10    | 8         | Sampler |
| Transformata Fourier Cuantică | 30Q              | 100% | Rată de succes (Procentul de rulări în care răspunsul corect este șirul de biți cu cel mai mare număr)      | 10    | 8        | Sampler |
| Estimarea Fazei Cuantice  | 30Q   | 99,9998%  | Acuratețea unghiului găsit: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Simulare cuantică: modelul Ising (15 pași) | 20Q  | 99,775%   |  $A$ (definit mai jos)  |  60 (per pas)  | 15 (per pas)   | Estimator |
| Simulare cuantică 2: dinamica moleculară (20 puncte de timp) | 34Q  |  96,78%  |  $A_{mean}$ (definit mai jos)   | 10 (per punct de timp)    | 6 (per punct de timp)  | Estimator |

Definind acuratețea măsurătorii unei valori de așteptare - metrica $A$ este definită astfel:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
unde $ \epsilon^{ideal} $ = valoarea de așteptare ideală,  $ \epsilon^{meas} $ = valoarea de așteptare măsurată, $\epsilon^{ideal}_{max} $ = valoarea maximă ideală, și $\epsilon^{ideal}_{min}$ = valoarea minimă ideală. $A_{mean}$ este pur și simplu media valorii $A$ pentru mai multe măsurători.

Această metrică este folosită deoarece este invariantă la deplasări globale și la scalarea în intervalul valorilor posibile. Cu alte cuvinte, indiferent dacă deplasezi intervalul valorilor posibile de așteptare mai sus sau mai jos ori mărești distribuția, valoarea lui $A$ ar trebui să rămână consistentă.
## Noțiuni de bază
Fire Opal Performance Management folosește Qiskit v`2.0.0`, care este versiunea recomandată. Versiunile acceptate sunt Qiskit >=v`2.0.0`.
Autentifică-te folosind [cheia ta API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează funcția Qiskit după cum urmează. (Acest fragment de cod presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul tău local.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Rulează Circuit-ul**

Rulează Circuit-ul și, opțional, definește Backend-ul și numărul de măsurători (shots).

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Poți folosi API-urile familiare [Qiskit Serverless](/guides/serverless) pentru a verifica starea lucrării funcției tale Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Recuperează rezultatul**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Rezultatele au același format ca un [rezultat Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Sampler primitive
### Exemplu Sampler
Folosește primitivul Sampler din Fire Opal Performance Management pentru a rula un circuit Bernstein–Vazirani. Acest algoritm, folosit pentru a găsi un șir ascuns din rezultatele unei funcții cutie neagră, este un algoritm comun de evaluare a performanței deoarece există un singur răspuns corect.
**1. Creează Circuit-ul**

Definește răspunsul corect al algoritmului, șirul de biți ascuns, și Circuit-ul Bernstein–Vazirani. Poți ajusta lățimea Circuit-ului schimbând pur și simplu `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Rulează Circuit-ul**

Rulează Circuit-ul și, opțional, definește Backend-ul și numărul de shots.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Verifică [statusul](/guides/functions#check-job-status) sarcinii de lucru a funcției Qiskit sau returnează [rezultatele](/guides/functions#retrieve-results) astfel:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Reprezintă grafic primele șiruri de biți**

Reprezintă grafic șirul de biți cu cel mai mare număr de apariții pentru a verifica dacă șirul ascuns a fost valoarea modală.